# Cross-Coders: Joint SAEs Across Activation Streams

This notebook demonstrates IRTK's `cross_coder` module for training a single
sparse autoencoder across multiple activation sources:

1. **CrossCoder class** – shared encoder, per-stream decoders
2. **Training** – joint training on multiple models/hooks
3. **Shared vs specific features** – which features generalize?
4. **Finetuning feature diff** – what changed between base and finetuned?
5. **Cross-layer crosscoder** – feature reuse across layers

## Why This Matters

CrossCoders train sparse autoencoders jointly on activations from two models (e.g., base and finetuned), discovering shared features and model-specific features. This is a powerful tool for understanding what changes during finetuning at the feature level.

**Key references:**
- [Bricken et al. (2023) "Towards Monosemanticity"](https://transformer-circuits.pub/2023/monosemantic-features/index.html) — Sparse autoencoders find interpretable features

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.cross_coder import (
    CrossCoder,
    train_crosscoder,
    shared_vs_specific_features,
    finetuning_feature_diff,
    cross_layer_crosscoder,
)

In [ ]:
# Build two models (simulating base and finetuned)
def make_model(seed=42):
    cfg = HookedTransformerConfig(
        n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50,
    )
    model = HookedTransformer(cfg)
    key = jax.random.PRNGKey(seed)
    leaves, treedef = jax.tree.flatten(model)
    new_leaves = []
    for leaf in leaves:
        if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
            key, subkey = jax.random.split(key)
            new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
        else:
            new_leaves.append(leaf)
    return jax.tree.unflatten(treedef, new_leaves)

base_model = make_model(seed=42)
ft_model = make_model(seed=99)  # different init simulates finetuning
seqs = [jnp.array([0, 1, 2, 3, 4, 5]), jnp.array([10, 11, 12, 13])]
print("Models ready: base (seed=42), finetuned (seed=99)")

## 1. CrossCoder Class

The CrossCoder learns a shared feature dictionary with per-stream decoders.

In [ ]:
cc = CrossCoder((16, 16), n_features=32, key=jax.random.PRNGKey(0))
print(f"Streams: {cc.n_streams}, Features: {cc.n_features}")
print(f"Encoder shape: {cc.W_enc.shape}")
print(f"Decoder 0 shape: {cc.W_decs[0].shape}")

# Forward pass
s1 = jnp.ones((4, 16))
s2 = jnp.ones((4, 16))
features = cc.encode([s1, s2])
print(f"\nEncoded features shape: {features.shape}")
out1, out2 = cc([s1, s2])
print(f"Decoded stream 0 shape: {out1.shape}")

## 2. Train a CrossCoder

Train jointly on activations from both models.

In [ ]:
hook = "blocks.0.hook_resid_post"
cc_trained = train_crosscoder(
    [base_model, ft_model], [hook, hook],
    seqs, n_features=32, n_steps=50, key=jax.random.PRNGKey(0),
)
print(f"Trained CrossCoder: {cc_trained.n_features} features, {cc_trained.n_streams} streams")

## 3. Shared vs Specific Features

Which features fire in both models (shared) vs one model only (specific)?

In [ ]:
svs = shared_vs_specific_features(
    cc_trained, [base_model, ft_model], [hook, hook], seqs
)
print(f"Shared features: {svs['n_shared']}")
print(f"Specific to stream 0 (base): {len(svs['specific_features'].get(0, []))}")
print(f"Specific to stream 1 (finetuned): {len(svs['specific_features'].get(1, []))}")
print(f"Sharing scores (first 8): {svs['sharing_scores'][:8].round(3)}")

## 4. Finetuning Feature Diff

What features were amplified, suppressed, or newly introduced by finetuning?

In [ ]:
diff = finetuning_feature_diff(
    base_model, ft_model, hook, seqs,
    n_features=32, n_steps=50, key=jax.random.PRNGKey(1),
)
print(f"Amplified features: {len(diff['amplified_features'])}")
print(f"Suppressed features: {len(diff['suppressed_features'])}")
print(f"Base-specific: {len(diff['base_specific'])}")
print(f"Finetuned-specific: {len(diff['finetuned_specific'])}")
print(f"Mean activation diff (first 8): {diff['mean_activation_diff'][:8].round(4)}")

## 5. Cross-Layer CrossCoder

Identify features reused across layers vs. layer-specific features.

In [ ]:
cl = cross_layer_crosscoder(
    base_model,
    ["blocks.0.hook_resid_post", "blocks.1.hook_resid_post"],
    seqs, n_features=32, n_steps=50, key=jax.random.PRNGKey(2),
)
print(f"Universal features: {len(cl['universal_features'])}")
print(f"Layer 0 specific: {len(cl['layer_specific_features'][0])}")
print(f"Layer 1 specific: {len(cl['layer_specific_features'][1])}")
print(f"Per-layer activity shape: {cl['per_layer_activity'].shape}")